# v10 Phase 2 — Re-Simulation with SL=10% / TP=60%

Breakeven drops from **27.3% → 14.3%** by widening TP and tightening SL.

This notebook re-runs the full option simulation (identical to v6 cell-5) with new exit rules:
- SL = −10% of entry premium (was −15%)
- TP = +60% of entry premium (was +40%)

Output: `sim_cache_v10.csv` — drop-in replacement for `v6/sim_cache.csv`.
Then open `backtest_v10.ipynb`, change `SIM_CACHE_PATH` to point here, and re-run.

**Estimated runtime: ~60–80 minutes** (same as v6 original simulation).

In [1]:
import pickle, warnings
import numpy as np
import pandas as pd
import yfinance as yf
from pathlib import Path
from datetime import date, timedelta

warnings.filterwarnings('ignore')

# ── ONLY CHANGE vs v6: SL and TP ─────────────────────────────────────────────
SL_PCT           = 0.10   # was 0.15  →  breakeven drops to 14.3%
TP_PCT           = 0.60   # was 0.40
BREAKEVEN        = SL_PCT / (SL_PCT + TP_PCT)

LOT_SIZE         = 75
STRIKE_STEP      = 50
BASE_LOTS        = 5
MAX_LOTS         = 25
DTE0_MAX_LOTS    = 10
STARTING_CAPITAL = 200_000.0
GAP_THR          = 0.0015
GAP_LARGE        = 0.0050
VIX_RISING_THR   = 0.03
VIX_SPIKE_THR    = 0.05
MAX_STALE_DAYS   = 5
OLD_CUTOFF       = date(2024, 11, 1)
_MON = ['JAN','FEB','MAR','APR','MAY','JUN','JUL','AUG','SEP','OCT','NOV','DEC']

print(f'SL={SL_PCT:.0%}  TP={TP_PCT:.0%}  Breakeven={BREAKEVEN:.1%}')
print(f'(v6 was SL=15%, TP=40%, breakeven=27.3% — now {BREAKEVEN:.1%})')

SL=10%  TP=60%  Breakeven=14.3%
(v6 was SL=15%, TP=40%, breakeven=27.3% — now 14.3%)


In [2]:
# ── Paths + data (identical to v6) ───────────────────────────────────────────
GAP_TRADING     = Path.cwd().parent
MARKET_RESEARCH = GAP_TRADING.parent
ALIGNED_CSV     = GAP_TRADING / 'v2' / 'v2_aligned_dataset.csv'
MINUTE_CACHE    = GAP_TRADING / 'kite_minute_cache'
MERGE_OLD       = MARKET_RESEARCH / 'merged' / 'old_format'
MERGE_NEW       = MARKET_RESEARCH / 'merged' / 'expiry_wise'

for lbl, p in [('aligned', ALIGNED_CSV), ('minute_cache', MINUTE_CACHE),
               ('merge_old', MERGE_OLD), ('merge_new', MERGE_NEW)]:
    print(f'{lbl:<14}: {"OK" if p.exists() else "MISSING"}')

aligned = pd.read_csv(ALIGNED_CSV, parse_dates=['india_date'])
aligned = aligned.sort_values('india_date').reset_index(drop=True)
aligned['VIX_INDIA_level'] = aligned['VIX_INDIA_level'].ffill().bfill()

# ── Spot map ──────────────────────────────────────────────────────────────────
spot_map = {}
for pkl_path in sorted(MINUTE_CACHE.glob('minute_256265_*.pkl')):
    with open(pkl_path, 'rb') as f:
        chunk = pickle.load(f)
    chunk.index = pd.to_datetime(chunk.index)
    if chunk.index.tzinfo is not None:
        chunk.index = chunk.index.tz_localize(None)
    for dt, row in chunk.iterrows():
        if dt.strftime('%H:%M') == '09:25':
            spot_map[dt.date()] = float(row['open'])
print(f'\nSpot map: {len(spot_map)} dates')

# ── N225 for signals ──────────────────────────────────────────────────────────
ds_start = aligned['india_date'].min().date() - timedelta(days=10)
ds_end   = aligned['india_date'].max().date() + timedelta(days=2)
try:
    n225 = yf.download('^N225', start=str(ds_start), end=str(ds_end), progress=False, auto_adjust=True)
    if isinstance(n225.columns, pd.MultiIndex): n225.columns = n225.columns.get_level_values(0)
    n225.index = pd.to_datetime(n225.index)
    if n225.index.tzinfo is None: n225.index = n225.index.tz_localize('UTC')
    n225_ok = len(n225) > 0
    print(f'N225: {len(n225)} rows')
except Exception as e:
    n225 = pd.DataFrame(); n225_ok = False; print(f'N225 failed: {e}')

def get_n225_sgx_ret(india_date):
    if not n225_ok: return None
    past = n225[n225.index.date < india_date]
    today = n225[n225.index.date == india_date]
    if len(past) < 2: return None
    cp = float(past['Close'].iloc[-1])
    if (india_date - past.index[-1].date()).days > MAX_STALE_DAYS: return None
    return (float(today['Open'].iloc[0]) - cp)/cp if len(today) > 0 else (cp - float(past['Close'].iloc[-2]))/float(past['Close'].iloc[-2])

aligned['sgx_ret_v5'] = aligned['india_date'].dt.date.map(get_n225_sgx_ret)
print(f'SGX ret: {aligned["sgx_ret_v5"].notna().sum()}/{len(aligned)} valid')

# ── NSE helpers + option loaders (copy from v6 — identical) ──────────────────
NSE_HOLIDAYS = {
    date(2024,1,22),date(2024,3,25),date(2024,3,29),date(2024,4,14),
    date(2024,4,17),date(2024,5,23),date(2024,6,17),date(2024,7,17),
    date(2024,8,15),date(2024,10,2),date(2024,10,24),date(2024,11,1),
    date(2024,11,15),date(2024,12,25),
    date(2025,2,26),date(2025,3,14),date(2025,3,31),date(2025,4,10),
    date(2025,4,14),date(2025,4,18),date(2025,5,1),date(2025,8,15),
    date(2025,8,27),date(2025,10,2),date(2025,10,21),date(2025,10,22),
    date(2025,11,5),date(2025,12,25),
    date(2026,1,26),date(2026,3,26),
}
EVENT_DAYS = {
    date(2024,2,1),date(2024,2,8),date(2024,4,5),date(2024,6,7),date(2024,8,8),
    date(2024,10,9),date(2024,12,6),date(2025,2,1),date(2025,2,7),date(2025,4,9),
    date(2025,6,6),date(2025,8,6),date(2025,10,8),date(2025,12,5),
    date(2026,2,1),date(2026,2,6),date(2026,4,8),date(2026,6,5),
    date(2026,8,7),date(2026,10,7),date(2026,12,4),
}
EXPIRY_CHANGE = date(2025, 9, 2)

def is_skip_day(d): return d.weekday()==0 or d in NSE_HOLIDAYS or d in EVENT_DAYS
def next_expiry(d):
    wd = 1 if d >= EXPIRY_CHANGE else 3
    return d + timedelta(days=(wd - d.weekday()) % 7)
def d2dmy(d): return f'{d.day:02d}{_MON[d.month-1]}{str(d.year)[2:]}'

def _load_old(trade_date, expiry_date):
    mon = _MON[trade_date.month-1]
    fp  = MERGE_OLD / f'2024{mon}' / f'NIFTY-{d2dmy(expiry_date)}-{d2dmy(trade_date)}.csv'
    if not fp.exists(): return None
    df = pd.read_csv(fp); df.columns = [c.strip() for c in df.columns]
    df['time_str'] = df['datetime'].astype(str).str[:5]; return df

def _load_new(trade_date, expiry_date):
    exp_dir = MERGE_NEW / expiry_date.strftime('%Y-%m-%d')
    if not exp_dir.exists(): return None
    exp_str = f'{expiry_date.day:02d}_{_MON[expiry_date.month-1]}_{str(expiry_date.year)[2:]}'
    files = list(exp_dir.glob(f'NIFTY_*_*_{exp_str}.csv'))
    if not files: return None
    dfs = []
    for fp in files:
        parts = fp.stem.split('_')
        try: strike = int(parts[1]); right = parts[2]
        except: continue
        try: df = pd.read_csv(fp, usecols=['timestamp','open','high','low','close','volume','oi'])
        except: continue
        df['ts'] = pd.to_datetime(df['timestamp']).dt.tz_localize(None)
        day = df[df['ts'].dt.date == trade_date].copy()
        if day.empty: continue
        day['time_str'] = day['ts'].dt.strftime('%H:%M')
        day['strike_price'] = strike; day['right'] = right
        day.rename(columns={'oi':'open_interest'}, inplace=True)
        dfs.append(day[['time_str','strike_price','right','open','high','low','close']])
    if not dfs: return None
    return pd.concat(dfs).sort_values(['time_str','strike_price','right']).reset_index(drop=True)

_opt_cache = {}
def load_opt(trade_date, expiry_date):
    k = (trade_date, expiry_date)
    if k not in _opt_cache:
        _opt_cache[k] = _load_old(trade_date,expiry_date) if expiry_date < OLD_CUTOFF else _load_new(trade_date,expiry_date)
    return _opt_cache[k]

print('Setup complete.')

aligned       : OK
minute_cache  : OK
merge_old     : OK
merge_new     : OK

Spot map: 742 dates
N225: 743 rows
SGX ret: 736/740 valid
Setup complete.


In [3]:
# ── Trade simulation (SL/TP from cell-1 config) ───────────────────────────────
def round_trip_charges(entry_prem: float, exit_prem: float, lots: int) -> float:
    buy_val  = entry_prem * lots * LOT_SIZE
    sell_val = exit_prem  * lots * LOT_SIZE
    brok  = 20.0 * 2
    stamp = 0.00003  * buy_val
    stt   = 0.000625 * sell_val
    exch  = 0.00053  * (buy_val + sell_val)
    sebi  = 0.000001 * (buy_val + sell_val)
    gst   = 0.18     * (brok + exch + sebi)
    return round(brok + stamp + stt + exch + sebi + gst, 2)

def simulate_trade_real(d) -> dict | None:
    spot_925 = spot_map.get(d)
    if spot_925 is None:
        return None
    atm    = round(spot_925 / STRIKE_STEP) * STRIKE_STEP
    strike = atm - STRIKE_STEP
    expiry = next_expiry(d)
    dte    = (expiry - d).days
    opt_df = load_opt(d, expiry)
    if opt_df is None:
        return None
    pe = opt_df[(opt_df['strike_price'] == strike) & (opt_df['right'] == 'PE')].copy()
    if pe.empty:
        pe = opt_df[(opt_df['strike_price'] == atm) & (opt_df['right'] == 'PE')].copy()
        if pe.empty:
            return None
        strike = atm
    pe = pe.sort_values('time_str').reset_index(drop=True)
    entry_row = pe[pe['time_str'] == '09:25']
    if entry_row.empty:
        return None
    entry_prem = float(entry_row.iloc[0]['open'])
    if entry_prem < 0.5:
        return None
    sl_px = entry_prem * (1 - SL_PCT)
    tp_px = entry_prem * (1 + TP_PCT)
    monitor = pe[(pe['time_str'] >= '09:26') & (pe['time_str'] <= '11:15')]
    exit_prem, exit_reason, exit_time = None, '11:15 exit', '11:15'
    for _, row in monitor.iterrows():
        lo, hi, t = float(row['low']), float(row['high']), row['time_str']
        if lo <= sl_px:
            exit_prem, exit_reason, exit_time = sl_px, 'Stop Loss', t
            break
        if hi >= tp_px:
            exit_prem, exit_reason, exit_time = tp_px, 'Target Hit', t
            break
    if exit_prem is None:
        exit_row = pe[pe['time_str'] == '11:15']
        if not exit_row.empty:
            exit_prem = float(exit_row.iloc[0]['close'])
        elif not monitor.empty:
            exit_prem = float(monitor.iloc[-1]['close'])
        else:
            exit_prem = entry_prem
    return {
        'expiry'    : expiry,
        'dte'       : dte,
        'strike'    : int(strike),
        'entry_prem': round(entry_prem, 2),
        'exit_prem' : round(exit_prem,  2),
        'pnl_pts'   : round(exit_prem - entry_prem, 2),
        'exit_reason': exit_reason,
        'exit_time' : exit_time,
    }

print(f'simulate_trade_real ready  (SL={SL_PCT:.0%}, TP={TP_PCT:.0%}, breakeven={BREAKEVEN:.1%})')

# ── Simulation cache ──────────────────────────────────────────────────────────
CACHE_PATH = Path.cwd() / 'sim_cache_v10.csv'

SIGNAL_NAMES = [
    'Gap Up', 'Gap Up Strong', 'Gap Down',
    'Prev India UP', 'Prev India DOWN',
    'US UP', 'US DOWN',
    'SGX UP', 'SGX DOWN',
    'DAX UP',
    'VIX Rising', 'VIX Falling', 'VIX Spike',
]

def compute_signals(row) -> dict:
    gap = float(row['gap_pct']) if pd.notna(row.get('gap_pct', float('nan'))) else 0.0
    def _f(col):
        v = row.get(col)
        return float(v) if pd.notna(v) else None
    sgx  = _f('sgx_ret_v5')
    sp5  = _f('SP500_ret')
    dax  = _f('DAX_ret')
    vix  = _f('VIX_US_ret')
    prev = _f('prev_india_ret')
    return {
        'Gap Up'         : gap >  GAP_THR,
        'Gap Up Strong'  : gap >  GAP_LARGE,
        'Gap Down'       : gap < -GAP_THR,
        'Prev India UP'  : prev is not None and prev > 0,
        'Prev India DOWN': prev is not None and prev < 0,
        'US UP'          : sp5  is not None and sp5  > 0,
        'US DOWN'        : sp5  is not None and sp5  < 0,
        'SGX UP'         : sgx  is not None and sgx  > 0,
        'SGX DOWN'       : sgx  is not None and sgx  < 0,
        'DAX UP'         : dax  is not None and dax  > 0,
        'VIX Rising'     : vix  is not None and vix  > VIX_RISING_THR,
        'VIX Falling'    : vix  is not None and vix  < 0,
        'VIX Spike'      : vix  is not None and vix  > VIX_SPIKE_THR,
    }

if CACHE_PATH.exists():
    print(f'Cache found — loading {CACHE_PATH.name} ...')
    sim_df = pd.read_csv(CACHE_PATH, parse_dates=['date'])
    sim_df['date'] = sim_df['date'].dt.date
    for s in SIGNAL_NAMES:
        sim_df[s] = sim_df[s].astype(bool)
    print(f'Loaded {len(sim_df)} days  ({sim_df["date"].min()} → {sim_df["date"].max()})')
else:
    print('No cache — simulating ALL available trading days (60–80 min)...')
    print('Progress: one dot per 10 days.')
    all_rows, no_data, n_done = [], 0, 0

    for _, row in aligned.iterrows():
        d = row['india_date'].date()
        if is_skip_day(d):
            continue
        sigs = compute_signals(row)
        res  = simulate_trade_real(d)
        n_done += 1
        if n_done % 10 == 0:
            print('.', end='', flush=True)
        if res is None:
            no_data += 1
            continue
        r = {
            'date'       : d,
            'win'        : res['exit_reason'] == 'Target Hit',
            'exit_reason': res['exit_reason'],
            'entry_prem' : res['entry_prem'],
            'exit_prem'  : res['exit_prem'],
            'dte'        : res['dte'],
        }
        r.update(sigs)
        all_rows.append(r)

    print(f'\nDone ({n_done} candidate days checked).')
    sim_df = pd.DataFrame(all_rows)
    sim_df.to_csv(CACHE_PATH, index=False)
    print(f'{len(sim_df)} days saved → {CACHE_PATH}')
    print(f'Missing option data skipped: {no_data}')

print(f'\nOverall win rate: {sim_df["win"].mean():.1%}  ({sim_df["win"].sum()} / {len(sim_df)} trades)')
print(sim_df['exit_reason'].value_counts().to_string())

simulate_trade_real ready  (SL=10%, TP=60%, breakeven=14.3%)
No cache — simulating ALL available trading days (60–80 min)...
Progress: one dot per 10 days.
.........................................................
Done (576 candidate days checked).
402 days saved → c:\Users\sayan\OneDrive\Desktop\Projects\03_Market_Research\market-research\gap_trading\v10\sim_cache_v10.csv
Missing option data skipped: 174

Overall win rate: 12.9%  (52 / 402 trades)
exit_reason
Stop Loss     322
Target Hit     52
11:15 exit     28


In [4]:
# ── Quarterly base TP rate analysis ──────────────────────────────────────────
# Shows whether the wider TP (60%) resolves bad quarters vs v6 breakeven=27.3%
sim_df['quarter'] = [f'{d.year}Q{(d.month-1)//3+1}' for d in sim_df['date']]

BE_V6  = 0.15 / (0.15 + 0.40)   # 27.3%  — old breakeven
BE_V10 = BREAKEVEN               # 14.3%  — new breakeven

q_stats = (sim_df.groupby('quarter')['win']
           .agg(N='count', TP_hits='sum')
           .assign(WR=lambda x: x['TP_hits'] / x['N'])
           .reset_index())
q_stats['Below_v6_BE']  = q_stats['WR'] < BE_V6
q_stats['Below_v10_BE'] = q_stats['WR'] < BE_V10

print('=' * 68)
print(f'  Quarterly base TP rate  (SL={SL_PCT:.0%} / TP={TP_PCT:.0%} / BE={BE_V10:.1%})')
print(f'  Compare: v6 breakeven={BE_V6:.1%}  |  v10 breakeven={BE_V10:.1%}')
print('=' * 68)
print(f'{"Quarter":<10} {"N":>4} {"TP hits":>7} {"WR":>7} {"<v6 BE":>8} {"<v10 BE":>9}')
print('-' * 68)
for _, r in q_stats.iterrows():
    flag_v6  = '*** BELOW ***' if r['Below_v6_BE']  else ''
    flag_v10 = '*** BELOW ***' if r['Below_v10_BE'] else ''
    print(f'{r["quarter"]:<10} {int(r["N"]):>4} {int(r["TP_hits"]):>7} '
          f'{r["WR"]:>6.1%}  {flag_v6:<13}  {flag_v10}')
print('=' * 68)

n_bad_v6  = q_stats['Below_v6_BE'].sum()
n_bad_v10 = q_stats['Below_v10_BE'].sum()
print(f'Quarters below breakeven:  v6={n_bad_v6}  →  v10={n_bad_v10}')
print(f'(v6 sim_cache overall WR: check v6/sim_cache.csv)')
print(f' v10 overall WR: {sim_df["win"].mean():.1%}')

print()
print('Exit reason breakdown:')
print(sim_df.groupby(['quarter', 'exit_reason'])['win']
      .count().unstack(fill_value=0).to_string())

  Quarterly base TP rate  (SL=10% / TP=60% / BE=14.3%)
  Compare: v6 breakeven=27.3%  |  v10 breakeven=14.3%
Quarter       N TP hits      WR   <v6 BE   <v10 BE
--------------------------------------------------------------------
2024Q1       47       8  17.0%  *** BELOW ***  
2024Q2       44       8  18.2%  *** BELOW ***  
2024Q3       46       4   8.7%  *** BELOW ***  *** BELOW ***
2024Q4       42       8  19.0%  *** BELOW ***  
2025Q1       48       7  14.6%  *** BELOW ***  
2025Q2       42       3   7.1%  *** BELOW ***  *** BELOW ***
2025Q3       49       5  10.2%  *** BELOW ***  *** BELOW ***
2025Q4       43       6  14.0%  *** BELOW ***  *** BELOW ***
2026Q1       41       3   7.3%  *** BELOW ***  *** BELOW ***
Quarters below breakeven:  v6=9  →  v10=5
(v6 sim_cache overall WR: check v6/sim_cache.csv)
 v10 overall WR: 12.9%

Exit reason breakdown:
exit_reason  11:15 exit  Stop Loss  Target Hit
quarter                                       
2024Q1                3         36       